# Predicción del Estado de la Carretera (PCI)
Este notebook carga los datos de PCI histórico, prepara las variables, entrena dos modelos y compara resultados.

Modelos incluidos:
1. Random Forest Regression (regresión tradicional)
2. Red neuronal con embeddings para variables categóricas

## 1. Instalación de dependencias
Si tu entorno no tiene `scikit-learn` o `tensorflow`, ejecuta esta celda.

In [44]:
!pip install scikit-learn tensorflow pandas matplotlib seaborn


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Cargar librerías y datos

In [45]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

sns.set_style('whitegrid')

file_path = '../CSVS/PCI-histo-cleaned-2005.csv'
df = pd.read_csv(file_path)
print('Datos cargados:', df.shape)
df.head()

Datos cargados: (203268, 10)


,CNN,PCI_Score,Previous_PCI,PCI_Change,Street_Name,PCI_Change_Date,Treatment_or_Survey,Functional_Class,Latitude,Longitude
0,100000,100.0,100.0,0.0,01st st,2005-11-16 16:50:05,Survey,Arterial,37.790794,-122.398869
1,100000,100.0,100.0,0.0,01st st,2007-07-26 16:21:27,Survey,Arterial,37.790794,-122.398869
2,100000,93.0,100.0,-7.0,01st st,2009-08-10 15:43:09,Survey,Arterial,37.790794,-122.398869
3,100000,73.0,93.0,-20.0,01st st,2010-12-23 13:37:35,Survey,Arterial,37.790794,-122.398869
4,100000,56.0,73.0,-17.0,01st st,2013-01-08 15:18:02,Survey,Arterial,37.790794,-122.398869


## 3. Exploración inicial
Revisamos columnas, tipos y valores faltantes para decidir qué features usar.

In [46]:
print('Columnas:', df.columns.tolist())
print('Tipos de datos:')
print(df.dtypes)
print('Valores nulos:')
print(df.isna().sum())
print('Categorías:')
print('Treatment_or_Survey:', df['Treatment_or_Survey'].nunique())
print('Functional_Class:', df['Functional_Class'].nunique())
print('Street_Name:', df['Street_Name'].nunique())

Columnas: ['CNN', 'PCI_Score', 'Previous_PCI', 'PCI_Change', 'Street_Name', 'PCI_Change_Date', 'Treatment_or_Survey', 'Functional_Class', 'Latitude', 'Longitude']
Tipos de datos:
CNN                      int64
PCI_Score              float64
Previous_PCI           float64
PCI_Change             float64
Street_Name                str
PCI_Change_Date            str
Treatment_or_Survey        str
Functional_Class           str
Latitude               float64
Longitude              float64
dtype: object
Valores nulos:
CNN                    0
PCI_Score              0
Previous_PCI           0
PCI_Change             0
Street_Name            0
PCI_Change_Date        0
Treatment_or_Survey    0
Functional_Class       0
Latitude               0
Longitude              0
dtype: int64
Categorías:
Treatment_or_Survey: 2
Functional_Class: 5
Street_Name: 4246


## 4. Selección de variables
Usaremos el siguiente conjunto de variables:
- `Previous_PCI` como variable numérica histórica
- `PCI_Change_Date` descompuesta en año y mes
- `Latitude`, `Longitude` como variables espaciales
- `Treatment_or_Survey` y `Functional_Class` como categóricas pequeñas
- `Street_Name` como categórica grande, tratada con embedding en el modelo neural

Target:
- `PCI_Score`

> No usamos `PCI_Change` como feature porque es una medida del cambio observado y, para predecir el estado futuro de la carretera, esa información no estaría disponible en el momento de la predicción.

In [47]:
df['PCI_Change_Date'] = pd.to_datetime(df['PCI_Change_Date'], errors='coerce')
df['pci_year'] = df['PCI_Change_Date'].dt.year
df['pci_month'] = df['PCI_Change_Date'].dt.month

# Agrupar los nombres de calle menos frecuentes como 'Other' para el embedding
top_streets = df['Street_Name'].value_counts().nlargest(200).index
df['Street_Name_top'] = df['Street_Name'].where(df['Street_Name'].isin(top_streets), 'Other')

features = [
    'Previous_PCI', 'Latitude', 'Longitude',
    'pci_year', 'pci_month',
    'Treatment_or_Survey', 'Functional_Class', 'Street_Name_top'
]
target = 'PCI_Score'

print('Features usados:', features)
print('Total de observaciones después de transformar fechas:', df.shape[0])

Features usados: ['Previous_PCI', 'Latitude', 'Longitude', 'pci_year', 'pci_month', 'Treatment_or_Survey', 'Functional_Class', 'Street_Name_top']
Total de observaciones después de transformar fechas: 203268


## 5. Preprocesamiento
Normalizamos variables numéricas y codificamos variables categóricas.
Para el modelo de regresión tradicional usamos OneHotEncoder.
Para la red neuronal usaremos embeddings en las categorías.

> Atención: los resultados anteriores eran sospechosamente buenos porque el conjunto de datos contiene muchos registros del mismo `CNN` (segmento) y el split aleatorio permite que el mismo segmento aparezca en entrenamiento y prueba. Por eso usamos `GroupShuffleSplit` sobre `CNN` para evaluar de forma más realista.


In [48]:
numeric_features = ['Previous_PCI', 'Latitude', 'Longitude', 'pci_year', 'pci_month']
categorical_features = ['Treatment_or_Survey', 'Functional_Class', 'Street_Name_top']

df_model = df[features + [target, 'CNN']].copy()
df_model = df_model.dropna(subset=features + [target])

X = df_model[features]
y = df_model[target]
groups = df_model['CNN']

group_split = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(group_split.split(X, y, groups=groups))
X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]
y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ],
    remainder='drop'
)

X_train_rf = preprocessor.fit_transform(X_train)
X_test_rf = preprocessor.transform(X_test)

print('Shape X_train para RandomForest:', X_train_rf.shape)
print('Shape X_test para RandomForest:', X_test_rf.shape)

Shape X_train para RandomForest: (162568, 213)
Shape X_test para RandomForest: (40700, 213)


## 6. Modelo 1: Random Forest Regression
Este modelo sirve como referencia tradicional para comparar con la red neuronal.

In [49]:
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train_rf, y_train)
y_pred_rf = rf.predict(X_test_rf)

mae_rf = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf = r2_score(y_test, y_pred_rf)

print('Random Forest Regression')
print(f'MAE: {mae_rf:.3f}')
print(f'RMSE: {rmse_rf:.3f}')
print(f'R2: {r2_rf:.4f}')

Random Forest Regression
MAE: 5.798
RMSE: 9.352
R2: 0.7547


## 7. Modelo 2: Red neuronal con embeddings
Construimos una red neuronal que utiliza embeddings para las variables categóricas.

In [50]:
# Preparar codificación ordinal para categorías y normalización numérica
ordinal_encoder = OrdinalEncoder()
X_train_cat = X_train[categorical_features].copy()
X_test_cat = X_test[categorical_features].copy()
X_train_cat = ordinal_encoder.fit_transform(X_train_cat)
X_test_cat = ordinal_encoder.transform(X_test_cat)

scaler = StandardScaler()
X_train_num = scaler.fit_transform(X_train[numeric_features])
X_test_num = scaler.transform(X_test[numeric_features])

# Definición de entradas para el modelo con embeddings
inputs = []
embeddings = []

for i, cat in enumerate(categorical_features):
    num_unique = int(df_model[cat].nunique()) + 1
    embed_dim = min(50, (num_unique + 1) // 2)
    input_cat = keras.Input(shape=(1,), name=f'{cat}_input')
    embed_cat = layers.Embedding(input_dim=num_unique, output_dim=embed_dim, name=f'{cat}_emb')(input_cat)
    embed_cat = layers.Reshape((embed_dim,))(embed_cat)
    inputs.append(input_cat)
    embeddings.append(embed_cat)

numeric_input = keras.Input(shape=(len(numeric_features),), name='numeric_input')
inputs.append(numeric_input)
embeddings.append(numeric_input)

x = layers.Concatenate()(embeddings)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.2)(x)
x = layers.Dense(64, activation='relu')(x)
x = layers.Dense(32, activation='relu')(x)
output = layers.Dense(1, activation='linear')(x)

model = keras.Model(inputs=inputs, outputs=output, name='pci_embedding_model')
model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()

# Preparar diccionarios de entrada para Keras
train_inputs = {f'{cat}_input': X_train_cat[:, i].astype('int32') for i, cat in enumerate(categorical_features)}
train_inputs['numeric_input'] = X_train_num.astype('float32')

test_inputs = {f'{cat}_input': X_test_cat[:, i].astype('int32') for i, cat in enumerate(categorical_features)}
test_inputs['numeric_input'] = X_test_num.astype('float32')

history = model.fit(
    train_inputs,
    y_train.values.astype('float32'),
    validation_split=0.15,
    epochs=15,
    batch_size=256,
    callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)],
    verbose=2
)

Model: "pci_embedding_model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ Treatment_or_Surve… │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Functional_Class_i… │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Street_Name_top_in… │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Treatment_or_Surve… │ (None, 1, 2)      │          6 │ Treatment_or_Sur… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Functional_Class_e… │ (None, 1, 3)      │         18 │ Functional_Class… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Street_Name_top_emb │ (None, 1, 50)     │     10,100 │ Street_Name_top_… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_12          │ (None, 2)         │          0 │ Treatment_or_Sur… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_13          │ (None, 3)         │          0 │ Functional_Class… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_14          │ (None, 50)        │          0 │ Street_Name_top_… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ numeric_input       │ (None, 5)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_4       │ (None, 60)        │          0 │ reshape_12[0][0], │
│ (Concatenate)       │                   │            │ reshape_13[0][0], │
│                     │                   │            │ reshape_14[0][0], │
│                     │                   │            │ numeric_input[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_16 (Dense)    │ (None, 128)       │      7,808 │ concatenate_4[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 128)       │          0 │ dense_16[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_17 (Dense)    │ (None, 64)        │      8,256 │ dropout_4[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_18 (Dense)    │ (None, 32)        │      2,080 │ dense_17[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_19 (Dense)    │ (None, 1)         │         33 │ dense_18[0][0]    │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 28,301 (110.55 KB)

 Trainable params: 28,301 (110.55 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/15
540/540 - 4s - 8ms/step - loss: 693.0855 - mae: 16.2123 - val_loss: 158.2618 - val_mae: 9.4410
Epoch 2/15
540/540 - 1s - 3ms/step - loss: 137.0365 - mae: 8.4810 - val_loss: 150.2060 - val_mae: 9.3854
Epoch 3/15
540/540 - 2s - 4ms/step - loss: 128.1892 - mae: 8.1174 - val_loss: 148.6547 - val_mae: 9.0289
Epoch 4/15
540/540 - 2s - 4ms/step - loss: 124.7638 - mae: 7.9371 - val_loss: 151.1593 - val_mae: 8.9065
Epoch 5/15
540/540 - 1s - 3ms/step - loss: 123.6470 - mae: 7.8808 - val_loss: 147.0527 - val_mae: 8.6043
Epoch 6/15
540/540 - 2s - 3ms/step - loss: 121.8353 - mae: 7.8001 - val_loss: 146.2841 - val_mae: 8.4032
Epoch 7/15
540/540 - 1s - 3ms/step - loss: 120.8060 - mae: 7.7449 - val_loss: 144.2695 - val_mae: 8.3547
Epoch 8/15
540/540 - 1s - 3ms/step - loss: 119.7519 - mae: 7.7065 - val_loss: 149.1111 - val_mae: 8.2704
Epoch 9/15
540/540 - 2s - 3ms/step - loss: 118.7548 - mae: 7.6499 - val_loss: 142.8478 - val_mae: 8.1334
Epoch 10/15
540/540 - 2s - 3ms/step - loss: 117.7539 -

In [51]:
# Evaluación del modelo con embeddings
y_pred_nn = model.predict(test_inputs).flatten()
mae_nn = mean_absolute_error(y_test, y_pred_nn)
rmse_nn = np.sqrt(mean_squared_error(y_test, y_pred_nn))
r2_nn = r2_score(y_test, y_pred_nn)

print('Red Neuronal con Embeddings')
print(f'MAE: {mae_nn:.3f}')
print(f'RMSE: {rmse_nn:.3f}')
print(f'R2: {r2_nn:.4f}')

1272/1272 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step
Red Neuronal con Embeddings
MAE: 7.238
RMSE: 10.610
R2: 0.6843


## 8. Comparación de modelos
Comparamos el desempeño entre el Random Forest y la red neuronal con embeddings.

In [52]:
results = pd.DataFrame({
    'Modelo': ['Random Forest', 'Red Neuronal (Embeddings)'],
    'MAE': [mae_rf, mae_nn],
    'RMSE': [rmse_rf, rmse_nn],
    'R2': [r2_rf, r2_nn],
})
results

,Modelo,MAE,RMSE,R2
0,Random Forest,5.798073,9.351667,0.754709
1,Red Neuronal (Embeddings),7.238467,10.609687,0.684276


## 9. Uso del modelo para predecir el futuro
Ejemplo de cómo preparar un nuevo registro y predecir el PCI.

In [53]:
sample = pd.DataFrame([
    {
        'Previous_PCI': 85.0,
        'Latitude': 37.78,
        'Longitude': -122.42,
        'pci_year': 2026,
        'pci_month': 6,
        'Treatment_or_Survey': 'Survey',
        'Functional_Class': 'Arterial',
        'Street_Name_top': 'Other'
    }
])

sample_rf = preprocessor.transform(sample)
sample_nn_cat = ordinal_encoder.transform(sample[categorical_features])
sample_nn_num = scaler.transform(sample[numeric_features])
sample_nn = {f'{cat}_input': sample_nn_cat[:, i].astype('int32') for i, cat in enumerate(categorical_features)}
sample_nn['numeric_input'] = sample_nn_num.astype('float32')

print('Predicción Random Forest:', rf.predict(sample_rf)[0])
print('Predicción Red Neuronal:', model.predict(sample_nn).flatten()[0])

Predicción Random Forest: 80.06
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
Predicción Red Neuronal: 80.79122


## 10. Conclusión
Hemos entrenado dos modelos distintos y preparado el procesamiento completo: normalización numérica y embeddings para categorías.
Puedes ajustar hiperparámetros, aumentar el tamaño del conjunto de embeddings o explorar más fechas para mejorar la predicción.

## Resumen de todo el procedimiento

### 1. Carga de datos

- Se carga el CSV `PCI-histo-cleaned-2005.csv` con `pandas`.
- Se imprime la forma del dataset, los tipos de columna y los valores nulos para conocer la calidad de los datos.

### 2. Exploración inicial

- Se revisan las columnas disponibles.
- Se comprueba si hay valores faltantes.
- Se examinan las categorías de `Treatment_or_Survey`, `Functional_Class` y `Street_Name`.

### 3. Ingeniería de variables

- Se convierte `PCI_Change_Date` a datetime.
- Se extraen `pci_year` y `pci_month`.
- Se elimina el uso de `pci_day` y `pci_dayofweek` porque el modelo solo debe usar mes y año.
- Se agrupan los `Street_Name` menos frecuentes como `Other` para simplificar los embeddings y reducir la dimensionalidad.

### 4. Selección de features

Variables usadas:

- Numéricas:
  - `Previous_PCI`
  - `Latitude`
  - `Longitude`
  - `pci_year`
  - `pci_month`
- Categóricas:
  - `Treatment_or_Survey`
  - `Functional_Class`
  - `Street_Name_top`
- Target:
  - `PCI_Score`

### 5. Prevención de fuga de datos

- Se detectó que `PCI_Change` era peligroso porque describe el cambio observado y no está disponible antes de predecir.
- Por eso se quitó `PCI_Change` de las features.
- El dataset contiene muchos registros del mismo `CNN` (segmento de calle).
- Para evitar evaluación engañosa, se usa `GroupShuffleSplit` con `groups=CNN`.
- Esto asegura que un mismo segmento no aparezca simultáneamente en entrenamiento y prueba.
- De lo contrario, el modelo puede memorizar el segmento y producir métricas demasiado optimistas.

### 6. Preprocesamiento

- `StandardScaler` normaliza las variables numéricas.
- `OneHotEncoder` codifica las variables categóricas para el modelo Random Forest.
- `ColumnTransformer` une las transformaciones en un solo pipeline:
  - `('num', numeric_transformer, numeric_features)`
  - `('cat', categorical_transformer, categorical_features)`
  - `remainder='drop'`
- Este pipeline es correcto y es la forma estándar de combinar transformaciones.

### 7. Modelo 1: Random Forest Regression

- Se entrena un `RandomForestRegressor` con los datos preprocesados.
- Se evalúa con:
  - `MAE`
  - `RMSE`
  - `R2`
- Este modelo sirve como referencia tradicional.

### 8. Modelo 2: Red neuronal con embeddings

- Se codifican las categorías con `OrdinalEncoder` para usarlas como índices en embeddings.
- Se normalizan los numéricos con `StandardScaler`.
- Se crea una red neuronal con:
  - entradas separadas para cada categoría
  - una capa `Embedding` para cada variable categórica
  - entrada numérica concatenada
  - capas densas intermedias (`128`, `64`, `32`)
  - salida lineal
- Se usa `EarlyStopping` para evitar sobreajuste.

### 9. Evaluación comparativa

- Se comparan los dos modelos con una tabla de métricas.
- Se muestra qué modelo tiene menor error y mejor ajuste.

### 10. Predicción de ejemplo

- Se incluye una celda de ejemplo con un nuevo registro.
- Ese registro se transforma con los mismos preprocessors.
- Se predice con ambos modelos.

### 11. Por qué los resultados anteriores eran sospechosos

- Un split aleatorio sin agrupar por `CNN` permite que el mismo segmento esté en entrenamiento y prueba.
- Esto puede inflar mucho métricas como `R2`.
- Por eso los valores muy buenos iniciales eran un indicio de evaluación poco realista.

### Resultado actual

- El procedimiento ahora es más riguroso y realista.
- El modelo ya no usa información que no estaría disponible al momento de predecir.
- El split por `CNN` es la mejora clave para evaluar mejor.

### Mejora opcional

- Se puede añadir una celda final que compare explícitamente:
  - split aleatorio vs split por `CNN`
- Así se vería claramente la diferencia en las métricas.
